# 00 - Setup Check on Kaggle

Mục tiêu: Kiểm tra phiên bản thư viện, khả năng sử dụng GPU/CUDA, duyệt và đếm số lượng ảnh/mask từng lớp trong dataset dưới `/kaggle/input`, kiểm tra tải trọng số ImageNet khi tắt internet, và ghi nhận dung lượng đĩa vào `setup_report.json`.


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
import torch
import torchvision

report = {
    "system": {},
    "gpu": {},
    "dataset": {},
    "weights_test": {},
    "disk_space": {}
}

print("=== 1. SYSTEM INFORMATION ===")
python_version = sys.version
torch_version = torch.__version__
torchvision_version = torchvision.__version__

print(f"Python Version: {python_version}")
print(f"PyTorch Version: {torch_version}")
print(f"Torchvision Version: {torchvision_version}")

report["system"] = {
    "python_version": python_version,
    "torch_version": torch_version,
    "torchvision_version": torchvision_version
}


In [ ]:
print("=== 2. GPU & CUDA INSPECTION ===")
cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)] if cuda_available else []

print(f"CUDA Available: {cuda_available}")
print(f"GPU Count: {gpu_count}")
for i, name in enumerate(gpu_names):
    print(f"  GPU {i}: {name}")

nvidia_smi_output = "N/A"
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    nvidia_smi_output = res.stdout
    print("\n--- nvidia-smi Output ---")
    print(nvidia_smi_output)
except Exception as e:
    nvidia_smi_output = f"Error running nvidia-smi: {str(e)}"
    print(nvidia_smi_output)

report["gpu"] = {
    "cuda_available": cuda_available,
    "gpu_count": gpu_count,
    "gpu_names": gpu_names,
    "nvidia_smi": nvidia_smi_output
}


In [ ]:
print("=== 3. DATASET INSPECTION UNDER /kaggle/input ===")
input_root = Path("/kaggle/input")

if not input_root.exists():
    print("WARNING: /kaggle/input does not exist! Running in non-Kaggle environment.")
    input_root = Path(".")

print(f"Inspecting root: {input_root.resolve()}")
top_level_items = list(input_root.iterdir())
print("Top-level items found:")
for item in top_level_items:
    print(f" - {item.name} ({'DIR' if item.is_dir() else 'FILE'})")

# Dynamically locate dataset folder
dataset_dir = None
for item in input_root.rglob("*"):
    if item.is_dir() and "covid" in item.name.lower() and "radiography" in item.name.lower():
        dataset_dir = item
        break

if dataset_dir is None and len(top_level_items) > 0:
    # fallback to first top level directory
    for item in top_level_items:
        if item.is_dir():
            dataset_dir = item
            break

print(f"\nDetected Dataset Directory: {dataset_dir}")

dataset_report = {
    "dataset_dir": str(dataset_dir),
    "classes": {},
    "total_images": 0,
    "total_masks": 0
}

valid_extensions = {'.png', '.jpg', '.jpeg'}

if dataset_dir and dataset_dir.exists():
    # Traverse children of dataset directory
    # Note: structure could be dataset_dir/COVID-19_Radiography_Dataset/{COVID, Normal, ...}
    # or directly dataset_dir/{COVID, Normal, ...}
    inner_dirs = [d for d in dataset_dir.iterdir() if d.is_dir()]
    if len(inner_dirs) == 1 and "radiography" in inner_dirs[0].name.lower():
        target_search_dir = inner_dirs[0]
    else:
        target_search_dir = dataset_dir
    
    print(f"Target search directory: {target_search_dir}")
    for class_folder in sorted(target_search_dir.iterdir()):
        if not class_folder.is_dir():
            continue
        class_name = class_folder.name
        images_count = 0
        masks_count = 0
        subdirs_found = [sub.name for sub in class_folder.iterdir() if sub.is_dir()]
        
        # check for 'images' subdirectory
        images_dir = class_folder / "images"
        if images_dir.exists() and images_dir.is_dir():
            images_count = sum(1 for f in images_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)
        else:
            # count image files directly in class_folder
            images_count = sum(1 for f in class_folder.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)
            
        # check for 'masks' subdirectory
        masks_dir = class_folder / "masks"
        if masks_dir.exists() and masks_dir.is_dir():
            masks_count = sum(1 for f in masks_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)
            
        dataset_report["classes"][class_name] = {
            "folder_name": class_name,
            "subdirectories": subdirs_found,
            "images_count": images_count,
            "masks_count": masks_count
        }
        dataset_report["total_images"] += images_count
        dataset_report["total_masks"] += masks_count
        
        print(f"Class folder: {class_name} | Subdirs: {subdirs_found} | Images: {images_count} | Masks: {masks_count}")
    
    print(f"\nTotal Images Counted: {dataset_report['total_images']}")
    print(f"Total Masks Counted: {dataset_report['total_masks']}")
else:
    print("Could not locate dataset folder.")

report["dataset"] = dataset_report


In [ ]:
print("=== 4. TEST TORCHVISION PRETRAINED WEIGHTS (INTERNET OFF) ===")
weights_status = "UNKNOWN"
error_msg = None

try:
    print("Attempting to instantiate resnet18 with ImageNet weights...")
    # try modern torchvision weights API
    model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    weights_status = "SUCCESS"
    print("SUCCESS: Model and weights loaded successfully.")
except Exception as e:
    weights_status = "FAILED"
    error_msg = str(e)
    print("TẢI TRỌNG SỐ THẤT BẠI (có thể do internet tắt)")
    print(f"Chi tiết lỗi: {error_msg}")

report["weights_test"] = {
    "status": weights_status,
    "error_message": error_msg
}


In [ ]:
print("=== 5. DISK SPACE INSPECTION ===")
working_dir = Path("/kaggle/working")
if not working_dir.exists():
    working_dir = Path(".")

total, used, free = shutil.disk_usage(working_dir)

total_gb = round(total / (2**30), 2)
used_gb = round(used / (2**30), 2)
free_gb = round(free / (2**30), 2)

print(f"Disk Usage for {working_dir.resolve()}:")
print(f"  Total: {total_gb} GB")
print(f"  Used:  {used_gb} GB")
print(f"  Free:  {free_gb} GB")

report["disk_space"] = {
    "working_dir": str(working_dir.resolve()),
    "total_gb": total_gb,
    "used_gb": used_gb,
    "free_gb": free_gb
}


In [ ]:
print("=== 6. WRITE /kaggle/working/setup_report.json ===")
output_report_path = Path("/kaggle/working/setup_report.json")
if not output_report_path.parent.exists():
    output_report_path = Path("setup_report.json")

with open(output_report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Report saved successfully to {output_report_path.resolve()}.")
print("\n--- setup_report.json Content Preview ---")
print(json.dumps(report, ensure_ascii=False, indent=2))
